In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from hft.utils.validate import plot_stats, numeric_stats, MDI, PFV

In [ ]:
# 数据导入
#ob = pd.read_csv('/Users/wook/Desktop/因子挖掘/raw_data/BTCUSDT_book_snapshot_5_20240528.csv.gz')
#ob = pd.read_csv('/Users/wook/Desktop/因子挖掘/processed_data_10s/BTCUSDT_book_snapshot_5_20240528_10s_orderbook.csv')
ob = pd.read_csv('/Users/wook/Downloads/因子挖掘/Sample data/raw/筛选BTC分段数据/book5_snapshot_1s/LONG_0001_MA做多_snapshot.csv')
tr = pd.read_csv('/Users/wook/Downloads/因子挖掘/Sample data/raw/筛选BTC分段数据/trade/LONG_0001_MA做多_trade.csv')
#tr = pd.read_csv('/Users/wook/Desktop/因子挖掘/processed_data_1s/BTCUSDT_trades_20240528_1s_trades.csv')


# 对于ob
ob = ob.rename(columns={
    'timestamp': 'ts',
    'bids[0].price': 'bp1', 'bids[0].amount': 'bv1',
    'asks[0].price': 'ap1', 'asks[0].amount': 'av1',
    'bids[1].price': 'bp2', 'bids[1].amount': 'bv2',
    'asks[1].price': 'ap2', 'asks[1].amount': 'av2',
    'bids[2].price': 'bp3', 'bids[2].amount': 'bv3',
    'asks[2].price': 'ap3', 'asks[2].amount': 'av3',
    'bids[3].price': 'bp4', 'bids[3].amount': 'bv4',
    'asks[3].price': 'ap4', 'asks[3].amount': 'av4',
    'bids[4].price': 'bp5', 'bids[4].amount': 'bv5',
    'asks[4].price': 'ap5', 'asks[4].amount': 'av5'
})

tr = tr.rename(columns={'timestamp':'ts', 'price':'p', 'amount':'v'})
tr['v'] = np.where(tr['side'] == 'sell', -tr['v'], tr['v'])

In [ ]:
ob

In [ ]:
tr

In [ ]:
import hft
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import time
from scipy.stats import pearsonr, zscore, spearmanr

from hft.utils.wrapper import trade_to_depth
from hft.utils.validate import plot_stats
from hft.utils.target.mid_price_changes import all_return
from hft.utils.target import filled_return, mid_price_changes, mp_changes
from hft.utils.combine import linear_model, SGDlinear_model
from hft.utils.format import purged_train_test_split, single_split
from hft.utils.format import depth_to_depth


from hft.signal.arrive_rate import arrive_rate
from hft.signal.cofi import cofi
from hft.signal.depth_age import depth_bid_age, depth_ask_age
from hft.signal.depth_changes import depth_bid_change, depth_ask_change
from hft.signal.fair_spread import fair_spread
from hft.signal.large_jump import large_jump
from hft.signal.llt import llt
from hft.signal.oir import oir
from hft.signal.order_flow import oflow
from hft.signal.price_distance import price_distance
from hft.signal.price_impact import price_impact
from hft.signal.return_lag import ask_lag,bid_lag
from hft.signal.swr import swr
from hft.signal.tick_vpin import tick_vpin
from hft.signal.volume_at_same_price  import volume_at_same_price
from hft.signal.volume import ask_volume, bid_volume
from hft.signal.weakoir import weakoir
from hft.signal.wss import wss

In [ ]:
Fre = 600
# 计算所有因子
arrive_rate_factor = arrive_rate(
    datas={'depth5': ob, 'trade': tr},
    params={'n': 600}  #用逐笔交易数据，分析n笔成交的平均时间
)

# 公平价差因子
cofi_factor = cofi(
    datas={'depth5': ob},
    params={}
)

# 深度账簿年龄因子
depth_bid_age_factor = depth_bid_age(
    datas={'depth5': ob},
    params={'n': Fre}
)
depth_ask_age_factor = depth_ask_age(
    datas={'depth5': ob},
    params={'n': Fre}
)

# 深度账簿变化因子
depth_bid_change_factor = depth_bid_change(
    datas={'depth5': ob},
    params={'n': Fre}
)
depth_ask_change_factor = depth_ask_change(
    datas={'depth5': ob},
    params={'n': Fre}
)

# 公平价差因子
fair_spread_factor = fair_spread(
    datas={'depth5': ob},
    params={}   
)

# 大跳因子（向上和向下）
large_jump_up_factor = large_jump(
    datas={'depth5': ob},
    params={'n': Fre, 'direct': 'up', 'jump_ratio': 0.001}
)

large_jump_down_factor = large_jump(
    datas={'depth5': ob},
    params={'n': 100, 'direct': 'down', 'jump_ratio': 0.001}
)

# 计算LLT平滑值
llt_factor = llt(
    datas={"depth5": ob},
    params={"n": Fre }
)

# 订单失衡比率
oir_factor = oir(
    datas={'depth5': ob},
    params={}
)

# 订单流因子
bid_order_flow_factor = oflow(
    datas={'depth5': ob},
    params={'side': 'bid', 'bend_ratio': 4}
)
ask_order_flow_factor = oflow(
    datas={'depth5': ob},
    params={'side': 'ask', 'bend_ratio': 4}
)


# 价格距离因子
ask_price_distance_factor = price_distance(
    datas={'depth5': ob},
    params={'n': Fre, 'side': 'ask'}
)
bid_price_distance_factor = price_distance(
    datas={'depth5': ob},
    params={'n': Fre, 'side': 'bid'}
)

# 价格冲击因子
price_impact_factor = price_impact(
    datas={'depth5': ob},
    params={'n': 5}
    
)

# price_swap_factor = price_swap(
#     datas={'depth5': ob},
#     params={}
# )

#滞后因子
ask_lag_factor = ask_lag(
    datas={'depth5': ob},
    params={'n': Fre}
)
bid_lag_factor = bid_lag(
    datas={'depth5': ob},
    params={'n': Fre}
)

#ser因子
ask_swr_factor = swr( 
    datas={'depth5': ob},
     params={'side': 'ask'}
)
bid_swr_factor = swr( 
    datas={'depth5': ob},
     params={'side': 'bid'}
)
#tick_vpin因子
tick_vpin_factor = tick_vpin(
    datas={'depth5': ob, 'trade': tr},
    params={'n': 600}
)

#volume_at_same_price因子
volume_at_same_price_factor = volume_at_same_price(
    datas={'depth5': ob, 'trade': tr},
    params={'n': 600}
)


# 交易量因子
ask_volume_factor = ask_volume(
    datas={'depth5': ob},
    params={}
)
bid_volume_factor = bid_volume(
    datas={'depth5': ob},
    params={}
)

# 弱订单失衡比率
weakoir_factor = weakoir(
    datas={'depth5': ob},
    params={}
)

#wss因子
wss_factor = wss(
    datas={'depth5': ob},
    params={'n':5}
)


# 合并所有因子为DataFrame
factors_df = pd.DataFrame({
    'arrive_rate': arrive_rate_factor,
    'cofi': cofi_factor,
    'depth_bid_age': depth_bid_age_factor,
    'depth_ask_age': depth_ask_age_factor,
    'depth_bid_change': depth_bid_change_factor,
    'depth_ask_change': depth_ask_change_factor,
    'fair_spread': fair_spread_factor,
    'large_jump_up': large_jump_up_factor,
    'large_jump_down': large_jump_down_factor,
    'llt': llt_factor,
    'oir': oir_factor,
    'bid_order_flow': bid_order_flow_factor,
    'ask_order_flow': ask_order_flow_factor,
    'ask_price_distance': ask_price_distance_factor,
    'bid_price_distance': bid_price_distance_factor,
    'price_impact': price_impact_factor,
    'weakoir': weakoir_factor,
    'ask_lag': ask_lag_factor,
    'bid_lag': bid_lag_factor,
    'ask_swr': ask_swr_factor,
    'bid_swr': bid_swr_factor,
    'tick_vpin': tick_vpin_factor,
     'ask_volume_at_same_price': volume_at_same_price_factor,
    'ask_volume': ask_volume_factor,
    'bid_volume': bid_volume_factor,
    'weakoir': weakoir_factor,
    'wss': wss_factor
    })

In [ ]:
# 创建标签数据
mid_price = (ob['ap1'] + ob['bp1']) / 2
ask_label = mid_price.diff(600).shift(-10).fillna(0)  
bid_label = -mid_price.diff(600).shift(-10).fillna(0)

In [ ]:
stats_full = numeric_stats({
    "datas": {
        "orderbook": ob,
        "trade": tr
    },
    "feature": arrive_rate_factor,
    "ask_label": ask_label,
    "bid_label": bid_label
})

print("\n完整统计指标:")
for key, value in stats_full.items():
    print(f"{key}: {value:.6f}")


In [ ]:
factors_df

In [ ]:
import os
import struct
import numpy as np
import pandas as pd

# 在你的特征计算代码末尾添加以下代码

# 设置保存路径
feature_path = "/Users/wook/Downloads/FactorTest/feature_test_data/feature"
label_path = "/Users/wook/Downloads/FactorTest/feature_test_data/label"
factor_name_path = "/Users/wook/Downloads/FactorTest/feature_error_test"

# 创建目录（如果不存在）
os.makedirs(feature_path, exist_ok=True)
os.makedirs(label_path, exist_ok=True)
os.makedirs(factor_name_path, exist_ok=True)

# 文件名
file_date = "20240528"

def save_binary_data(data, filepath):
    """
    将numpy数组保存为二进制文件
    """
    with open(filepath, 'wb') as f:
        for value in data.flatten():
            if np.isnan(value) or np.isinf(value):
                value = 0.0  # 将NaN和Inf替换为0
            f.write(struct.pack('d', float(value)))

def calculate_labels(ob_data, lookforward_periods=10):
    """
    计算标签 - 使用未来价格变化作为标签
    """
    # 计算中间价
    mid_price = (ob_data['bp1'] + ob_data['ap1']) / 2
    
    # 计算未来收益率作为标签
    future_returns = mid_price.shift(-lookforward_periods) / mid_price - 1
    
    # 填充最后几个NaN值为0
    future_returns = future_returns.fillna(0)
    
    return future_returns.values

# 1. 计算标签
print("计算标签...")
labels = calculate_labels(ob, lookforward_periods=600)  # 10期后的收益率
print(f"标签数量: {len(labels)}")
print(f"标签统计: min={np.nanmin(labels):.6f}, max={np.nanmax(labels):.6f}, mean={np.nanmean(labels):.6f}")

# 2. 处理特征数据
print("处理特征数据...")
# 确保所有特征长度一致，取最短长度
min_length = min(len(labels), len(factors_df))
factors_df_aligned = factors_df.iloc[:min_length].copy()
labels_aligned = labels[:min_length]

# 填充NaN值
factors_df_aligned = factors_df_aligned.fillna(0)

print(f"对齐后的数据长度: {min_length}")
print(f"特征数量: {len(factors_df_aligned.columns)}")

# 3. 保存特征名称文件
print("保存特征名称...")
factor_names = [f"{name}.csv" for name in factors_df_aligned.columns]
factor_names_df = pd.DataFrame(factor_names, columns=['factor_name'])
factor_names_path = os.path.join(factor_name_path, "factorname.csv")
factor_names_df.to_csv(factor_names_path, header=False, index=False)
print(f"特征名称已保存到: {factor_names_path}")

# 4. 保存特征数据为二进制文件
print("保存特征数据...")
feature_file_path = os.path.join(feature_path, file_date)

# 将DataFrame转换为numpy数组，按列优先顺序排列
feature_values = []
for col in factors_df_aligned.columns:
    feature_values.extend(factors_df_aligned[col].values)

feature_array = np.array(feature_values)
save_binary_data(feature_array, feature_file_path)
print(f"特征数据已保存到: {feature_file_path}")
print(f"特征文件大小: {len(feature_array) * 8} 字节")

# 5. 保存标签数据为二进制文件
print("保存标签数据...")
label_file_path = os.path.join(label_path, file_date)
save_binary_data(labels_aligned, label_file_path)
print(f"标签数据已保存到: {label_file_path}")
print(f"标签文件大小: {len(labels_aligned) * 8} 字节")

# 6. 验证保存的数据
print("\n验证保存的数据...")

# 验证特征文件
with open(feature_file_path, 'rb') as f:
    feature_data_check = f.read()
n_feature_values = len(feature_data_check) // 8
feature_check = struct.unpack(f'{n_feature_values}d', feature_data_check)
print(f"特征文件验证 - 读取数值数量: {len(feature_check)}")
print(f"特征文件验证 - 数值范围: [{min(feature_check):.6f}, {max(feature_check):.6f}]")

# 验证标签文件
with open(label_file_path, 'rb') as f:
    label_data_check = f.read()
n_label_values = len(label_data_check) // 8
label_check = struct.unpack(f'{n_label_values}d', label_data_check)
print(f"标签文件验证 - 读取数值数量: {len(label_check)}")
print(f"标签文件验证 - 数值范围: [{min(label_check):.6f}, {max(label_check):.6f}]")

# 验证特征名称文件
factor_names_check = pd.read_csv(factor_names_path, header=None)
print(f"特征名称文件验证 - 特征数量: {len(factor_names_check)}")
print(f"前5个特征名称: {factor_names_check.iloc[:5, 0].tolist()}")

print("\n数据保存完成！")
print(f"数据摘要:")
print(f"- 时间点数量: {min_length}")
print(f"- 特征数量: {len(factors_df_aligned.columns)}")
print(f"- 特征数据文件: {feature_file_path}")
print(f"- 标签数据文件: {label_file_path}")
print(f"- 特征名称文件: {factor_names_path}")

# 7. 显示如何运行分析脚本
print(f"\n运行特征分析的命令:")
print(f"python feature_analysis.py \\")
print(f"  --feature_base_path '{feature_file_path}' \\")
print(f"  --label_file '{label_file_path}' \\")
print(f"  --feature_name '{factor_names_path}' \\")
print(f"  --output_base_path '/Users/wook/Downloads/FactorTest/feature_analysis_output'")